In [1]:
pip install pylint

In [2]:
import pandas as pd
import numpy as np
import nbformat
import re
import tokenize
from pygments import lex
from pygments.lexers import PythonLexer
from pygments.token import Token
from nltk.tokenize import RegexpTokenizer
from io import BytesIO
from pylint.lint import Run

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
users_code_location_df = pd.read_csv('/content/drive/MyDrive/thesis/part1 data-extraction/1.4 filtered-df-extraction/gender_undersampled_codes.csv', index_col=0)

users_code_location_df['code_location'] = users_code_location_df['code_location'].apply(lambda path: '/content/drive/MyDrive/thesis/notebooks/' + "/".join(path.split('\\')[3:]))

users_code_location_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2000 entries, 0 to 1999
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Username       2000 non-null   object
 1   Display_Name   2000 non-null   object
 2   Gender         2000 non-null   object
 3   notebook_url   2000 non-null   object
 4   code_location  2000 non-null   object
 5   labels         2000 non-null   object
 6   top_labels     2000 non-null   object
dtypes: object(7)
memory usage: 125.0+ KB


In [5]:
def parse_notebook(notebook_content, notebook_celltype=None):
    code_cells = []
    for cell in notebook_content.cells:
        if not notebook_celltype or cell.cell_type == notebook_celltype:
            if cell.source:
                parsed_code = '\n'.join(line for line in cell.source.split('\n') if not line.strip().startswith('%'))
                code_cells.append(parsed_code)

    return code_cells

In [6]:
def count_tokens(code_sections):

    tokens = set()
    token_pattern = r'[A-Za-z0-9]+'
    tokenizer = RegexpTokenizer(token_pattern)

    for code_section in code_sections :

      try:
        tokens.update(tokenizer.tokenize(code_section))
      except Exception as e:
          print(code_section)
          print("Exception is",e)

    return len(set(tokens))

In [7]:
def char_count(parsed_code, char):
    return parsed_code.count(char)

In [8]:
def saved_words_count(code_sections, saved_word):
  count = 0

  for code_section in code_sections:

    tokens = list(lex(code_section, PythonLexer()))

    count += sum(1 for token_type, token_value in tokens if token_type is Token.Keyword and token_value == saved_word)

  return count

In [9]:
def calculate_comment_density(code_sections):

    total_lines = 0
    comment_lines = 0

    for code_section in code_sections :

      try:
          byte_stream = BytesIO(code_section.encode('utf-8'))

          for token in tokenize.tokenize(byte_stream.readline):
              if token.type == tokenize.NEWLINE or token.type == tokenize.NL:
                  total_lines += 1
              elif token.type == tokenize.COMMENT:
                  comment_lines += 1
      except Exception as e:
          print("Exception is",e)

    single_line_comment_density = comment_lines / total_lines if total_lines else 0

    return single_line_comment_density

In [10]:
def calculate_comment_tokens_density(code_content):
    lines = code_content.split('\n')

    comment_words = 0
    code_words = 0

    for line in lines:
        words = line.split()
        if line.strip().startswith("#"):
            comment_words += len(words)
        else:
            code_words += len(words)

    total_words = comment_words + code_words

    comment_density = comment_words / total_words if total_words > 0 else 0.0

    return comment_density

In [11]:
def extract_naming_from_code(code_sections):

    names_set = set()

    for code_section in code_sections:

      tokens = list(lex(code_section, PythonLexer()))

      names_set.update({token_value for token_type, token_value in tokens if token_type == Token.Name})

    return names_set

In [12]:
def extract_code_sections(ipynb_file_path):

    print("Starting extracting features for ",ipynb_file_path)

    with open(ipynb_file_path, 'r', encoding='utf-8') as notebook_file:
      notebook_content = nbformat.read(notebook_file, as_version=4)

    code_sections = parse_notebook(notebook_content, 'code')
    markdown_sections = parse_notebook(notebook_content, 'markdown')
    all_sections = parse_notebook(notebook_content)
    only_code_in_code_sections = [" \n ".join([row for row in code_section.split('\n') if not (row.startswith('#') and row.startswith('%'))]) for code_section in code_sections]

    number_of_lines = np.sum([code_section.count('\n') + 1 for code_section in code_sections])
    names_set = extract_naming_from_code(code_sections)

    num_of_sections = len(code_sections)
    token_count = count_tokens(code_sections)

    variables_count = len(names_set)
    function_count = saved_words_count(code_sections, "def")
    loop_count = saved_words_count(code_sections, "for")
    condition_count = saved_words_count(code_sections, "if") + saved_words_count(code_sections, "while")

    function_density = saved_words_count(code_sections, "def") / token_count if token_count > 0 else 0
    loop_density = saved_words_count(code_sections, "for") / token_count if token_count > 0 else 0
    condition_density = (saved_words_count(code_sections, "if") + saved_words_count(code_sections, "while")) / token_count if token_count > 0 else 0
    single_line_comment_density = calculate_comment_density(code_sections)
    comment_tokens_density = calculate_comment_tokens_density(" \n ".join(code_sections))

    features = [code_sections, markdown_sections, all_sections, only_code_in_code_sections,
                number_of_lines, names_set, num_of_sections, token_count,
                variables_count, function_count, loop_count, condition_count,
                single_line_comment_density,
                function_density, loop_density, condition_density, comment_tokens_density]

    print("Finished extracting features for ",ipynb_file_path)

    return features

In [13]:
users_code_location_df[['code_sections', 'markdown_sections', 'all_sections', 'only_code_in_code_sections',
                'number_of_lines', 'names_set', 'num_of_sections', 'token_count',
                'variables_count', 'function_count', 'loop_count', 'condition_count',
                'single_line_comment_density', 'function_density', 'loop_density',
                'condition_density', 'comment_tokens_density']] = users_code_location_df['code_location'].apply(extract_code_sections).apply(pd.Series)

Starting extracting features for  /content/drive/MyDrive/thesis/notebooks/male/tchaye59/jmarket-keras-starter-submit.ipynb
Finished extracting features for  /content/drive/MyDrive/thesis/notebooks/male/tchaye59/jmarket-keras-starter-submit.ipynb
Starting extracting features for  /content/drive/MyDrive/thesis/notebooks/female/iyara1/deepanalysis-and-modelling-using-linear-regression.ipynb
Finished extracting features for  /content/drive/MyDrive/thesis/notebooks/female/iyara1/deepanalysis-and-modelling-using-linear-regression.ipynb
Starting extracting features for  /content/drive/MyDrive/thesis/notebooks/male/sanjay7013/credit-card-fraud-detection-easy-to-understand.ipynb
Finished extracting features for  /content/drive/MyDrive/thesis/notebooks/male/sanjay7013/credit-card-fraud-detection-easy-to-understand.ipynb
Starting extracting features for  /content/drive/MyDrive/thesis/notebooks/female/validmodel/master-decisiontree-from-scratch-with-iris.ipynb
Finished extracting features for  /co

/usr/local/lib/python3.10/dist-packages/nbformat/__init__.py:96: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)


Finished extracting features for  /content/drive/MyDrive/thesis/notebooks/female/sadafpj/mobile-price-predition.ipynb
Starting extracting features for  /content/drive/MyDrive/thesis/notebooks/male/neerajmohan/nlp-bidirectional-lstm-using-tensorflow-2-0.ipynb
Finished extracting features for  /content/drive/MyDrive/thesis/notebooks/male/neerajmohan/nlp-bidirectional-lstm-using-tensorflow-2-0.ipynb
Starting extracting features for  /content/drive/MyDrive/thesis/notebooks/male/tunguz/image-resizing-96x96-test.ipynb
Finished extracting features for  /content/drive/MyDrive/thesis/notebooks/male/tunguz/image-resizing-96x96-test.ipynb
Starting extracting features for  /content/drive/MyDrive/thesis/notebooks/male/kutaykutlu/tpus-resnet50-leaf-disease.ipynb
Finished extracting features for  /content/drive/MyDrive/thesis/notebooks/male/kutaykutlu/tpus-resnet50-leaf-disease.ipynb
Starting extracting features for  /content/drive/MyDrive/thesis/notebooks/male/satishgunjal/advanced-reg-techniques-li

In [14]:
users_code_location_df.head()

,Username,Display_Name,Gender,notebook_url,code_location,labels,top_labels,code_sections,markdown_sections,all_sections,...,token_count,variables_count,function_count,loop_count,condition_count,single_line_comment_density,function_density,loop_density,condition_density,comment_tokens_density
0,tchaye59,Jude TCHAYE,male,https://www.kaggle.com/code/tchaye59/jmarket-k...,/content/drive/MyDrive/thesis/notebooks/male/t...,"['Jane Street Market Prediction', 'Jane Street...",{'Jane Street Market Prediction'},[# This Python 3 environment comes with many h...,[### This notebook is only dedicated to submit...,[# This Python 3 environment comes with many h...,...,155,30,0,3,0,0.333333,0.000000,0.019355,0.000000,0.524272
1,iyara1,Riya,female,https://www.kaggle.com/code/iyara1/deepanalysi...,/content/drive/MyDrive/thesis/notebooks/female...,['House Prices - Advanced Regression Techniques'],{'House Prices - Advanced Regression Techniques'},[import numpy as np\nimport pandas as pd\nimpo...,[**#Bivariate Analysis******],[import numpy as np\nimport pandas as pd\nimpo...,...,374,102,1,7,7,0.129825,0.002674,0.018717,0.018717,0.359568
2,sanjay7013,Sanjay M,male,https://www.kaggle.com/code/sanjay7013/credit-...,/content/drive/MyDrive/thesis/notebooks/male/s...,['Credit Card Fraud Detection'],{'Credit Card Fraud Detection'},[# This Python 3 environment comes with many h...,"[# Credit Card Fraud Detection, ### DataSet In...","[# Credit Card Fraud Detection, ### DataSet In...",...,206,65,0,2,0,0.166667,0.000000,0.009709,0.000000,0.386076
3,validmodel,Rashmi Margani,female,https://www.kaggle.com/code/validmodel/master-...,/content/drive/MyDrive/thesis/notebooks/female...,['Iris Species'],{'Iris Species'},[# This Python 3 environment comes with many h...,[# <h1 style='background:#f0c2c1; border:2; bo...,[# This Python 3 environment comes with many h...,...,302,95,12,9,12,0.159420,0.039735,0.029801,0.039735,0.279314
4,rajeevnair676,Rajeev Nair,male,https://www.kaggle.com/code/rajeevnair676/nlp-...,/content/drive/MyDrive/thesis/notebooks/male/r...,"['IMDB Dataset of 50K Movie Reviews', 'Tweet S...",{'Natural Language Processing with Disaster Tw...,[#Importing NLTK package\nimport nltk\n\nimpor...,[# <center><b> NLP STARTERS - PART 1 <center/>...,[# <center><b> NLP STARTERS - PART 1 <center/>...,...,197,63,5,17,3,0.095652,0.025381,0.086294,0.015228,0.193370


In [15]:
users_code_location_df.to_csv('/content/drive/MyDrive/thesis/part2 feature-creation/2.1 feature creation/code_sections_extracts.csv', index=False)